## Problem Introduction: Safe Reinforcement Learning

<img src="https://raw.githubusercontent.com/eleurent/highway-env/gh-media/docs/media/roundabout-env.gif"></img>


**Optimal Planning with Oracle Model**
When planning with the true model, an optimal policy generally operates as close as possible to the system constraints, resulting in dangerous behaviours.

Example of unsafe behaviour:
<a href="https://imgur.com/o5JvJXi"><img src="https://i.imgur.com/o5JvJXi.png" title="source: imgur.com" /></a>

When turning the green car takes a right but does not stay on the extreme right of the road. It follows the policy to not get hit but this is not safe since other cars may not behave optimally always.

As an effect, even slight model errors can lead to catastrophic failures

<a href="https://imgur.com/sIkfUCC"><img src="https://i.imgur.com/sIkfUCC.png" title="source: imgur.com" /></a>
  
In order to account for model uncertainty, we follow the robust control framework:

"*Maximise the worst-case performance with respect to a set of possible behavioral models*"

**Robust Planning with Discrete Ambiguity**

You can plan by considering every possible direction the traffic participants can take at their next intersection.


**Reducing uncertainty of the next action of other cars**

Let us discuss the reasons behind the uncertainty

1. Driver on the road is not an optimal driver.
2. We do not know the exact location of the driver.

<!-- **Handling Scenario 1:**
We plan by considering every possible direction that traffic participants can take at their next intersection.

$\mu$- mean path of each participant.

<a href="https://imgur.com/hARWjSD"><img src="https://i.imgur.com/hARWjSD.png" title="source: imgur.com" /></a>

Continuous range of trajectories (variance over) path trajectories of each traffic participant based on their driving style.

<a href="https://imgur.com/qd3kyZ0"><img src="https://i.imgur.com/qd3kyZ0.png" title="source: imgur.com" /></a> -->

To summarise, if everyone has the same driving behaviour we can assume drivers to behave as if they are driving on the optimal path. If we also think about each one having a driving behaviour i.e (some can be drunk, talking on phone, angry) then we get varying levels of uncertainty around the optimal path for each vehicle. This is accurately characterized below.

<a href="https://imgur.com/qfDGiTx"><img src="https://i.imgur.com/qfDGiTx.png" title="source: imgur.com" /></a>



# Model-Based Reinforcement Learning

We first demonstrate a Model based approach and its failure cases and then moivate you to explore inverse reinforcement learning approaches. You can choose either of the above approaches.

## Principle for Model based RL
We consider the optimal control problem of an MDP with a **known** reward function $R$ and subject to **unknown deterministic** dynamics $s_{t+1} = f(s_t, a_t)$:

$$\max_{(a_0,a_1,\dotsc)} \sum_{t=0}^\infty \gamma^t R(s_t,a_t)$$

In **model-based reinforcement learning**, this problem is solved in **two steps**:
1. **Model learning**:
We learn a model of the dynamics $f_\theta \simeq f$ through regression on interaction data.
2. **Planning**:
We leverage the dynamics model $f_\theta$ to compute the optimal trajectory $$\max_{(a_0,a_1,\dotsc)} \sum_{t=0}^\infty \gamma^t R(\hat{s}_t,a_t)$$ following the learnt dynamics $\hat{s}_{t+1} = f_\theta(\hat{s}_t, a_t)$.

(We can easily extend to unknown rewards and stochastic dynamics, but we consider the simpler case in this notebook for ease of presentation)


## Motivation

### Sparse rewards
* In model-free reinforcement learning, we only obtain a reinforcement signal when encountering rewards. In environment with **sparse rewards**, the chance of obtaining a reward randomly is **negligible**, which prevents any learning.
* However, even in the **absence of rewards** we still receive a **stream of state transition data**. We can exploit this data to learn about the task at hand.

### Complexity of the policy/value vs dynamics:
Is it easier to decide which action is best, or to predict what is going to happen?
* Some problems can have **complex dynamics** but a **simple optimal policy or value function**. For instance, consider the problem of learning to swim. Predicting the movement requires understanding fluid dynamics and vortices while the optimal policy simply consists in moving the limbs in sync.
* Conversely, other problems can have **simple dynamics** but **complex policies/value functions**. Think of the game of Go, its rules are simplistic (placing a stone merely changes the board state at this location) but the corresponding optimal policy is very complicated.

Intuitively, model-free RL should be applied to the first category of problems and model-based RL to the second category.

### Inductive bias
Oftentimes, real-world problems exhibit a particular **structure**: for instance, any problem involving motion of physical objects will be **continuous**. It can also be **smooth**, **invariant** to translations, etc. This knowledge can then be incorporated in machine learning models to foster efficient learning. In contrast, there can often be **discontinuities** in the policy decisions or value function: e.g. think of a collision vs near-collision state.

###  Sample efficiency
Overall, it is generally recognized that model-based approaches tend to **learn faster** than model-free techniques (see e.g. [[Sutton, 1990]](http://papersdb.cs.ualberta.ca/~papersdb/uploaded_files/paper_p160-sutton.pdf.stjohn)).

### Interpretability
In real-world applications, we may want to know **how a policy will behave before actually executing it**, for instance for **safety-check** purposes. However, model-free reinforcement learning only recommends which action to take at current time without being able to predict its consequences. In order to obtain the trajectory, we have no choice but executing the policy. In stark contrast, model-based methods a more interpretable in the sense that we can probe the policy for its intended (and predicted) trajectory.

## Our challenge: Robust

We consider the **roundabout-v0** task of the [highway-env](https://github.com/eleurent/highway-env) environment. It is a **continuous control** task where an agent **drives a car** by controlling the gaz pedal and steering angle and must **navigate safely** with the appropriate heading.



###  Warming up
We start with a few useful installs and imports:

In [1]:
# --- Fix NumPy and PyTorch Compatibility ---
!pip install docopt

# --- Install dev branch of highway-env ---
!pip install --user git+https://github.com/eleurent/highway-env

# Fix sys.path to include --user installs
import sys
sys.path.append("/root/.local/lib/python3.11/site-packages")

  Preparing metadata (setup.py) ... done
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=1a05e8f366c296ff4894e320cd469c5b104fa3482d5d495a424ca2c27d9a0385
  Stored in directory: /root/.cache/pip/wheels/1a/b0/8c/4b75c4116c31f83c8f9f047231251e13cc74481cca4a78a9ce
Successfully built docopt
  Cloning https://github.com/eleurent/highway-env to /tmp/pip-req-build-hi3ti4bh
  Running command git clone --filter=blob:none --quiet https://github.com/eleurent/highway-env /tmp/pip-req-build-hi3ti4bh
  Resolved https://github.com/eleurent/highway-env to commit c7c12cc30bd5a255cb846d7f7a875589bbf3684e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for highway-env: filename=highway_env-1.10.1-py3-none-any.whl size=106144 sha256=e28c9ad7d3961d83ab70bb3d698c5d66f7304ef22bf3d39d506ec863f6204fb9
  Stored in directory: /tmp/pip-ephem-wheel-cache-wr_uc2de/wheels/af/

In [2]:
# --- Clone rl-agents repo ---
%cd /content
%rm -rf /content/rl-agents
!git clone https://github.com/eleurent/rl-agents.git

# --- Patch logger.py and evaluation.py ---
from pathlib import Path

# Patch logger.py

logger_path = Path("/content/rl-agents/rl_agents/trainer/logger.py")
logger_text = logger_path.read_text()
# Replace default argument value
logger_text = logger_text.replace("gym_level=gym.logger.INFO", "gym_level=__import__('logging').INFO")
# Remove any line that tries to set the gym logger level
logger_lines = logger_text.splitlines()
logger_lines = [line for line in logger_lines if "gym.logger.set" not in line]
# Write back the patched file
logger_path.write_text("\n".join(logger_lines))

# Patch evaluation.py: remove broken import and define capped_cubic_video_schedule
eval_path = Path("/content/rl-agents/rl_agents/trainer/evaluation.py")
eval_text = eval_path.read_text()


eval_text = eval_text.replace("np.infty", "np.inf")
eval_text = eval_text.replace(
    "from gymnasium.wrappers import RecordVideo, RecordEpisodeStatistics, capped_cubic_video_schedule",
    "from gymnasium.wrappers import RecordVideo, RecordEpisodeStatistics"
)

inject_fn = """

def capped_cubic_video_schedule(episode_id):
    return True
"""

eval_lines = eval_text.splitlines()
for i, line in enumerate(eval_lines):
    if not line.strip().startswith("import") and not line.strip().startswith("from"):
        inject_index = i
        break
eval_lines = eval_lines[:inject_index] + [inject_fn.strip()] + eval_lines[inject_index:]
eval_path.write_text("\n".join(eval_lines))



/content
Cloning into 'rl-agents'...
remote: Enumerating objects: 6735, done.
remote: Counting objects: 100% (118/118), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 6735 (delta 42), reused 89 (delta 24), pack-reused 6617 (from 1)
Receiving objects: 100% (6735/6735), 1.03 MiB | 8.06 MiB/s, done.
Resolving deltas: 100% (4751/4751), done.


16612

In [3]:
import gymnasium as gym
import highway_env

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from collections import namedtuple

# Visualization
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
from tqdm.notebook import trange

# IO
from pathlib import Path

We also define a simple helper function for visualization of episodes:

In [4]:
import sys
from tqdm.notebook import trange
!pip install tensorboardx gym pyvirtualdisplay
!apt-get install -y xvfb ffmpeg

%cd /content
!git clone https://github.com/Farama-Foundation/HighwayEnv.git 2> /dev/null
%cd /content/HighwayEnv/scripts/
from utils import record_videos, show_videos

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 4.0 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
xvfb is already the newest version (2:21.1.4-2ubuntu1.7~22.04.14).
0 upgraded, 0 newly installed, 0 to remove and 34 not upgraded.
/content
/content/HighwayEnv/scripts


### Let's try it!

Make the environment, and run an episode with random actions:

In [5]:
env = gym.make("roundabout-v0", render_mode="rgb_array")
env = record_videos(env)
env.reset()
done = False
it = 0
while not done:
    it = it+1
    # This could be stuck in a bad path, kill it if it goes on for too long.
    if it == 100:
        print("Killed simulation, possibly stuck")
        break
    action = env.action_space.sample()
    obs, reward, done, truncated, info = env.step(action)
env.close()
show_videos()

The environment is a `GoalEnv`, which means the agent receives a dictionary containing both the current `observation` and the `desired_goal` that conditions its policy.

In [6]:
print("Observation format:", obs)

Observation format: [[ 1.0000000e+00  9.7754523e-02  2.4345967e-01  1.8480243e-01
  -3.9420629e-01]
 [ 1.0000000e+00  6.6740163e-02  2.2292590e-01  6.8684143e-01
  -2.7309183e-02]
 [ 1.0000000e+00  1.9853711e-01  3.5503665e-03  1.2810023e-01
  -1.0000000e+00]
 [ 1.0000000e+00  9.0614241e-01 -2.0000000e-02 -9.2418092e-01
   2.2204460e-16]
 [ 0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00
   0.0000000e+00]]


There is also an `achieved_goal` that won't be useful here (it only serves when the state and goal spaces are different, as a projection from the observation to the goal space).

Alright! We are now ready to apply the inverse reinforcement learning paradigm.

## Experience collection
First, we randomly interact with the environment to produce a batch of experiences

$$D = \{s_t, a_t, s_{t+1}\}_{t\in[1,N]}$$

In [7]:
Transition = namedtuple('Transition', ['state', 'action', 'next_state'])

def collect_interaction_data(env, size=5, action_repeat=2):
# def collect_interaction_data(env, size=1000, action_repeat=2):
    data, done = [], True
    for _ in trange(size, desc="Collecting interaction data"):
        action = env.action_space.sample()
        for _ in range(action_repeat):
            if done:
              previous_obs, info = env.reset()
            obs, reward, done, truncated, info = env.step(action)
            # print("obs:", obs)
            # print("reward:", reward)
            # print("previous_obs:", previous_obs)
            # break
            data.append(Transition(torch.Tensor(previous_obs),
                                   torch.Tensor(action),
                                   torch.Tensor(obs)))
            previous_obs = obs
        # break
    return data

env = gym.make("roundabout-v0")
data = collect_interaction_data(env)
env.close()
print("Sample transition:", data[0])

Sample transition: Transition(state=tensor([[ 1.0000e+00,  2.0000e-02,  4.5000e-01,  0.0000e+00, -5.3333e-01],
        [ 1.0000e+00, -5.4376e-02,  1.9247e-01,  7.4913e-01,  2.1164e-01],
        [ 1.0000e+00, -2.0474e-01,  1.2523e-01,  3.6804e-01,  6.0170e-01],
        [ 1.0000e+00,  1.0000e+00, -2.0000e-02, -9.2451e-01,  2.2204e-16],
        [ 1.0000e+00, -1.6191e-01, -1.1741e-01, -5.2352e-01,  7.2197e-01]]), action=tensor([0., 0., 0.]), next_state=tensor([[ 1.0000e+00,  3.2601e-02,  3.3082e-01,  1.8643e-01, -9.5755e-01],
        [ 1.0000e+00, -1.2997e-01,  1.9903e-01,  5.5925e-01,  4.2980e-01],
        [ 1.0000e+00,  6.0816e-02,  1.8770e-01,  7.5980e-01, -1.6935e-01],
        [ 1.0000e+00, -1.9735e-01,  9.2338e-03, -4.3669e-02,  8.9073e-01],
        [ 1.0000e+00,  1.0000e+00, -2.0000e-02, -9.2451e-01,  2.2204e-16]]))


In [8]:
%cd /content/rl-agents/
! pip install -e .

from pathlib import Path
import json

# This is to fix a mistake in the script that points to older location for robust

# Root directory where configs live
config_root = Path("/content/rl-agents/scripts/configs")

# Traverse all .json files recursively
for json_path in config_root.rglob("*.json"):
    try:
        content = json.loads(json_path.read_text())
        content_str = json.dumps(content)

        if "rl_agents.agents.tree_search.robust" in content_str:
            # Replace all references to the outdated path
            content_str = content_str.replace(
                "rl_agents.agents.tree_search.robust",
                "rl_agents.agents.robust.robust"
            )
            # Write back the updated content
            json_path.write_text(json.dumps(json.loads(content_str), indent=2))
            print(f"✔️ Patched: {json_path}")
    except Exception as e:
        print(f"⚠️ Skipped {json_path}: {e}")



/content/rl-agents
Obtaining file:///content/rl-agents
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 63.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.8

In [9]:
%cd /content/rl-agents/scripts/
! python experiments.py benchmark configs/RoundaboutEnv/benchmark_robust_control.json \
                      --test --episodes=10 --processes=1

/content/rl-agents/scripts
[INFO] Episode 0 score: 11.0 
[INFO] Episode 1 score: 10.9 
multiprocessing.pool.RemoteTraceback: 
"""
Traceback (most recent call last):
  File "/usr/lib/python3.11/multiprocessing/pool.py", line 125, in worker
    result = (True, func(*args, **kwds))
                    ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/multiprocessing/pool.py", line 51, in starmapstar
    return list(itertools.starmap(args[0], args[1]))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/rl-agents/scripts/experiments.py", line 79, in evaluate
    evaluation.test()
  File "/content/rl-agents/rl_agents/trainer/evaluation.py", line 138, in test
    self.run_episodes()
  File "/content/rl-agents/rl_agents/trainer/evaluation.py", line 150, in run_episodes
    reward, terminal = self.step()
                       ^^^^^^^^^^^
  File "/content/rl-agents/rl_agents/trainer/evaluation.py", line 182, in step
    transition = self.wrapped_env.step(action)
               

## Build a dynamics model

Dynamics model depends on the nature of environment. Model based RL suffers from the problem of **Model bias**

The model will be accurate only on some region of the state space that was explored and covered in  $D$ . Outside of  $D$ , the model may diverge and hallucinate important rewards. This effect is problematic when the model is used by a planning algorithm, as the latter will try to exploit these hallucinated high rewards and will steer the agent towards unknown (and thus dangerous) regions where the model is erroneously optimistic.

You need to choose from the following:
1. Model-based RL methods
2. Model-free RL methods

## Part 1

List all the known methods you have studied in the class from 1 and 2 and suggest which is the best method you wish to implement for this problem.


1. What is the method? Model-based or Model-free
2. Motivation for choosing the method
3. Problem formulation using your proposed method. Mention metrics (cost function), block-diagram of your pipeline with proper notation and mathematical formulation


## Part 2

Show implementation of your approach and mention results with plots. Add a section to showcase failure cases and mention limitation and future work.


**Example of a dynamics model**

Here is an sample example that uses a model based approach.

We now design a model to represent the system dynamics. We choose  a **structured model** inspired from *Linear Time-Invariant (LTI) systems*

$$\dot{x} = f_\theta(x, u) = A_\theta(x, u)x + B_\theta(x, u)u$$

where the $(x, u)$ notation comes from the Control Theory community and stands for the state and action $(s,a)$. Intuitively, we learn at each point $(x_t, u_t)$ the **linearization** of the true dynamics $f$ with respect to $(x, u)$.

We parametrize $A_\theta$ and $B_\theta$ as two fully-connected networks with one hidden layer.


In [12]:
class DynamicsModel(nn.Module):
    STATE_X = 0
    STATE_Y = 1

    def __init__(self, state_size, action_size, hidden_size, dt):
        super().__init__()
        self.state_size, self.action_size, self.dt = state_size, action_size, dt
        A_size, B_size = state_size * state_size, state_size * action_size
        self.A1 = nn.Linear(state_size + action_size, hidden_size)
        self.A2 = nn.Linear(hidden_size, A_size)
        self.B1 = nn.Linear(state_size + action_size, hidden_size)
        self.B2 = nn.Linear(hidden_size, B_size)

    def forward(self, x, u):
        """
            Predict x_{t+1} = f(x_t, u_t)
        :param x: a batch of states
        :param u: a batch of actions
        """
        print(x.shape, u.shape)
        xu = torch.cat((x, u), -1)
        xu[:, self.STATE_X:self.STATE_Y+1] = 0  # Remove dependency in (x,y)
        A = self.A2(F.relu(self.A1(xu)))
        A = torch.reshape(A, (x.shape[0], self.state_size, self.state_size))
        B = self.B2(F.relu(self.B1(xu)))
        B = torch.reshape(B, (x.shape[0], self.state_size, self.action_size))
        dx = A @ x.unsqueeze(-1) + B @ u.unsqueeze(-1)
        return x + dx.squeeze()*self.dt

dynamics_model = DynamicsModel(state_size=env.observation_space.shape[0],
                         action_size=env.action_space.n,
                         hidden_size=64,
                         dt=1/env.unwrapped.config["policy_frequency"])
# print("Forward initial model on a sample transition:", dynamics(data[0].state.unsqueeze(0),
#                                                                 data[0].action.unsqueeze(0)).detach())

## Scenario1: Leverage dynamics model for planning

We now use the learnt dynamics model $f_\theta$ for planning.
In order to solve the optimal control problem, we use a sampling-based optimization algorithm: the **Cross-Entropy Method** (`CEM`). It is an optimization algorithm applicable to problems that are both **combinatorial** and **continuous**, which is our case: find the best performing sequence of actions.

This method approximates the optimal importance sampling estimator by repeating two phases:
1. **Draw samples** from a probability distribution. We use Gaussian distributions over sequences of actions.
2. Minimize the **cross-entropy** between this distribution and a **target distribution** to produce a better sample in the next iteration. We define this target distribution by selecting the top-k performing sampled sequences.

![Credits to Olivier Sigaud](https://github.com/yfletberliac/rlss2019-hands-on/blob/master/imgs/cem.png?raw=1)

Note that as we have a local linear dynamics model, we could instead choose an `Iterative LQR` planner which would be more efficient. We prefer `CEM` in this educational setting for its simplicity and generality.

## Visualize a few episodes

En voiture, Simone !

In [13]:
env = gym.make("roundabout-v0", render_mode='rgb_array')
env = record_videos(env)
obs, info = env.reset()

# Sample cem_planner
def cem_planner(x0, action_dim, model, planning_horizon=15, iterations=5, samples=500, topk=50):
    # TO FILL
    # Like https://pytorch.org/rl/0.6/reference/generated/torchrl.modules.CEMPlanner.html
    return None  # First action


num_episodes = 100
episode_rewards = []

for episode in trange(num_episodes, desc=f"Testing {num_episodes} episodes..."):
    obs, info = env.reset()
    total_reward = 0

    done, truncated = False, False
    while not (done or truncated):
        action = cem_planner(
            torch.tensor(obs, dtype=torch.float32),
            env.action_space.n,
            model=dynamics_model
        )
        obs, reward, done, truncated, info = env.step(action.numpy())
        total_reward += reward

    episode_rewards.append(total_reward)

env.close()
show_videos()


/usr/local/lib/python3.11/dist-packages/gymnasium/wrappers/rendering.py:283: UserWarning: WARN: Overwriting existing videos at /content/rl-agents/scripts/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Testing 100 episodes...:   0%|          | 0/100 [00:00<?, ?it/s]

AttributeError: 'NoneType' object has no attribute 'numpy'

In [ ]:
print(type(obs), obs)


<class 'numpy.ndarray'> [[ 1.0000000e+00  2.0000000e-02  4.4999999e-01  0.0000000e+00
  -5.3333336e-01]
 [ 1.0000000e+00 -5.6871977e-02  1.9174352e-01  1.0000000e+00
   3.2133132e-01]
 [ 1.0000000e+00 -2.0365393e-01  1.2698455e-01  6.4733797e-01
   1.0000000e+00]
 [ 1.0000000e+00  1.0000000e+00 -2.0000000e-02 -1.0000000e+00
   2.2204460e-16]
 [ 1.0000000e+00 -1.7144071e-01 -1.0299556e-01 -5.3544241e-01
   8.9126778e-01]]


# Mention Limitations of your approach

### Example: Computational cost of planning

At test time, the planning step typically requires **sampling a lot of trajectories** to find a near-optimal candidate, wich may turn out to be very costly. This may be prohibitive in a high-frequency real-time setting. The **model-free** methods which directly recommend the best action are **much more efficient** in that regard.